# Trunchi v3: mai multe vii, convenție reparată

**De ce:** pe Vairão (Portugalia, vie nevăzută) modelele de acum găsesc trunchiul în doar **74 din 460** de poze.
65% din pozele de antrenare ale lui v2 vin din label map, unde trunchiul e un „ciot” tăiat de marginea de jos a pozei.

**Ce schimbă v3:**
1. label map redus la **30%** din pozele de antrenare;
2. **Vairão** adăugat (6 înregistrări la antrenare, 1 pentru alegerea epocii, 2 păstrate ca test);
3. etichete automate pe EscaYard B7 **mai stricte** (sigure + regăsite în oglindă; pozele nesigure sunt lăsate afară);
4. clasa „stâlp” etichetată automat cu YOLOE, ca modelul să nu mai confunde stâlpii cu trunchiurile;
5. epoca e aleasă pe **via-țintă** (cele 20 de poze B7 marcate de voi + o înregistrare Vairão), nu pe label map.

**Teste finale (niciodată la antrenare):** Vairão 170318 + 170442, testul label map (robot), EscaYard **B9**,
frunziș fără trunchi (alarme false). La final: tabel **v2 vs v3**.

## Ce faceți voi (totul la început)
1. **Runtime → Change runtime type → T4 GPU → Save**, apoi **Runtime → Run all**.
2. Celula 3: „Connect to Google Drive”.
3. Celula 4: dacă `model_trunchi_v2.pt` nu e în Drive, alegeți-l de pe laptop (`models/`).
4. Celula 5: lipiți cheia API Roboflow.

Durată estimată (nemăsurată pe acest set): **~2–3 ore**. Rezultatul: `MyDrive/vie_hackathon/trunk_v3_runs/model_trunchi_v3.pt`.

## 1. Placa video

In [ ]:
!nvidia-smi

## 2. Instalare

In [ ]:
!pip install -q ultralytics==8.4.163 py7zr
import ultralytics
print("ultralytics", ultralytics.__version__)

## 3. Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/vie_hackathon'
R = f"{OUT}/trunk_v3_runs"
os.makedirs(R, exist_ok=True)
os.environ["TRUNK_ROOT"] = "/content/tv"
os.environ["TRUNK_RUNS"] = R
for d in ("data/escayard/photos", "data/mildew", "data/vairao", "models"):
    os.makedirs(f"/content/tv/{d}", exist_ok=True)

## 4. Modelul v2 (punctul de plecare și profesorul pentru etichetele automate)

In [ ]:
import shutil
from google.colab import files

if os.path.exists(f"{OUT}/model_trunchi_v2.pt"):
    shutil.copy(f"{OUT}/model_trunchi_v2.pt", "/content/tv/models/model_trunchi_v2.pt")
while not os.path.exists("/content/tv/models/model_trunchi_v2.pt"):
    print("model_trunchi_v2.pt is not in Drive -> choose it from the laptop (folder models/)")
    for name, data in files.upload().items():
        if "model_trunchi_v2" in name:
            open("/content/tv/models/model_trunchi_v2.pt", "wb").write(data)
            shutil.copy("/content/tv/models/model_trunchi_v2.pt", f"{OUT}/model_trunchi_v2.pt")
print("OK: model_trunchi_v2.pt")

## 5. label map v8 (Roboflow) — lipiți cheia API

In [ ]:
import json, urllib.request, zipfile
from getpass import getpass

key = getpass("Roboflow API key: ")
info = json.load(urllib.request.urlopen(f"https://api.roboflow.com/grape-detection/label-map/8/yolov8?api_key={key}"))
urllib.request.urlretrieve(info["export"]["link"], "/content/labelmap.zip")
zipfile.ZipFile("/content/labelmap.zip").extractall("/content/tv/data/labelmap")
print({s: len(os.listdir(f"/content/tv/data/labelmap/{s}/images")) for s in ("train", "valid", "test")})

## 6. Celelalte dataseturi, toate deodată (~10–20 min): HUMAIN, Vairão, EscaYard, mană

In [ ]:
%%bash
cd /content
dl() {  # dl <name> <command>: runs in the background, writes dl_<name>.log ending with OK or FAILED
  ( bash -c "$2" > dl_$1.log 2>&1 && echo OK >> dl_$1.log || echo FAILED >> dl_$1.log ) &
}
# -o = overwrite without asking (a second run would otherwise wait for an answer and fail); --retry = retry on network errors
dl humain 'curl -sSfL --retry 3 -o humain.zip "https://github.com/humain-lab/vine-trunk/raw/master/Humain-Lab-vine-trunk-dataset.zip" && unzip -q -o humain.zip -d /content/tv/data/humain'
dl vairao 'curl -sSfL --retry 3 -o vairao.zip "https://zenodo.org/api/records/18152298/files/Dataset_Vairao_Grapevine_Trunks.zip/content" && unzip -q -o vairao.zip -d /content/tv/data/vairao'
dl escayard 'curl -sSfL --retry 3 -o /content/tv/data/escayard/datasheet.csv "https://zenodo.org/api/records/10362568/files/datasheet.csv/content" && curl -sSfL --retry 3 -o escayard.7z "https://zenodo.org/api/records/10362568/files/PLANTs_SmartPhonePhotos_JPG.7z/content" && python3 -c "import py7zr; py7zr.SevenZipFile(\"escayard.7z\").extractall(\"/content/tv/data/escayard/photos\")"'
dl mildew 'cd /content/tv/data/mildew && curl -sSfL --retry 3 -A "Mozilla/5.0" -o m.zip "https://data.mendeley.com/public-files/datasets/yxf2yv5ymt/files/baa9a436-bd95-425a-8d6f-aca7152cb1b0/file_downloaded" && unzip -q -o m.zip'
wait
for n in humain vairao escayard mildew; do
  echo "$n: $(tail -1 dl_$n.log)"
  if [ "$(tail -1 dl_$n.log)" != OK ]; then echo "   error:"; tail -4 dl_$n.log | head -3 | sed 's/^/   /'; fi
done

In [ ]:
import glob
counts = {
    "HUMAIN train photos": len(glob.glob("/content/tv/data/humain/git/train/Original_photos/*.jpg")),
    "Vairão frames": len(glob.glob("/content/tv/data/vairao/**/*.png", recursive=True)),
    "EscaYard photos": len(glob.glob("/content/tv/data/escayard/photos/**/*.jpg", recursive=True)),
    "mildew photos": len(glob.glob("/content/tv/data/mildew/downy_mildew_images/*.jpg")),
}
print(counts)
assert all(counts.values()), "a download failed: run the cell above again"

## 7. Scriptul (același ca în proiect: `tools/trunk_v3.py`) + cele 20 de poze B7 marcate de voi

In [ ]:
%%writefile /content/trunk_v3.py
"""Trunk model v3 for unseen vineyards: the data mix is rebalanced so the model stops learning "trunk = the stub at the
bottom edge of the photo" (label map, 65% of v2's training photos, 98% of its trunk boxes touch the bottom edge).

What changed from v2 (research: reports/Îmbunătățire modele ciorchini și trunchi.md, measured 27 Sept):
  - label map cut to at most 30% of the training photos
  - Vairão (Portugal, whole trunks seen from low, autumn) added: 6 recordings train, 1 validation, 2 test.
    v1+v2 find a trunk in only 74/460 Vairão photos, so this is the vineyard the current models do not know.
  - EscaYard B7 auto-labels are stricter: a box is kept only if v2 is sure (conf >= 0.4) and finds it again on the
    mirrored photo; a photo with an unsure box (0.25-0.4) is left out, because a missing box teaches "trunk = background"
  - posts (class 1 "stalp") auto-labeled by YOLOE with the prompt "concrete post" (tested: finds real posts,
    confidence 0.15-0.35), so posts become explicit negatives for the trunk class
  - checkpoint chosen on photos from the target vineyards (20 hand-marked B7 photos + Vairão recording 170105), not on
    label map; every epoch is saved and compared
Not used: the Spanish pruning set (Voxel51/vineyard-pruning): its "trunk" outline also covers the horizontal arms
(median box width 0.80 of the photo), a third trunk convention.

Final tests (never trained on): Vairão recordings 170318 + 170442, label map test (robot), EscaYard B9 (photos with a
trunk), mildew odd photos (foliage without trunks: false alarms).

Steps (the same file runs in Colab: TRUNK_ROOT = folder with data/ and models/, TRUNK_RUNS = where runs go):
  python -m tools.trunk_v3 build             # ~20-30 min on a T4 (auto-labels B7 + posts)
  python -m tools.trunk_v3 train --name v3   # fine-tune from models/model_trunchi_v2.pt, every epoch saved
  python -m tools.trunk_v3 select --run v3   # every epoch on the validation photos -> best epoch + threshold
  python -m tools.trunk_v3 report --weights <best.pt> --conf 0.3   # v2 vs v3 on the final tests
"""
import argparse
import csv
import json
import os
import random
import shutil
import time
from pathlib import Path

ROOT = Path(os.environ.get("TRUNK_ROOT", Path(__file__).resolve().parent.parent))
RUNS = Path(os.environ.get("TRUNK_RUNS", ROOT / "runs" / "trunk_v3"))
DS = ROOT / "data" / "trunk_v3"
LM, HUM, VAI = ROOT / "data" / "labelmap", ROOT / "data" / "humain" / "git", ROOT / "data" / "vairao"
ESCA, MILDEW = ROOT / "data" / "escayard", ROOT / "data" / "mildew" / "downy_mildew_images"
MANUAL = ROOT / "data" / "manual_trunks_b7.json"
V2 = ROOT / "models" / "model_trunchi_v2.pt"

MAX_SIDE = 1280  # every photo is stored at most this big: phone photos are 4000 px and would slow training down
LM_SHARE = 0.30  # label map photos: at most this share of the training photos
VAI_VALID, VAI_TEST = {"170105"}, {"170318", "170442"}  # Vairão recordings kept out of training
POST_PROMPTS = ["wooden post", "metal pole", "concrete post", "vine trunk", ""]  # "" = background, absorbs doubtful boxes
POST_CONF = 0.15
# Auto-label thresholds, chosen on the 20 hand-marked B7 photos (27 Sept): v2 is rarely very sure on EscaYard
# (median box conf 0.43). sure 0.5 kept 17/113 B7 photos; sure 0.4 + unsure 0.25 keeps 50/113, and on the hand-marked
# photos 7 of its 8 auto-boxes were real trunks.
SURE, UNSURE, MIRROR = 0.4, 0.25, 0.25
TEST_SETS = ("test_vairao", "test_lm", "test_foliage")


# ---------- geometry (boxes are (x1, y1, x2, y2), normalized 0-1 unless said otherwise) ----------

def iou(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def inside(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    smaller = min((a[2] - a[0]) * (a[3] - a[1]), (b[2] - b[0]) * (b[3] - b[1]))
    return inter / smaller if smaller > 0 else 0.0


def read_labels(path, classes):
    """YOLO boxes ("c x y w h") or outlines ("c x1 y1 x2 y2 ...") -> normalized xyxy boxes of the wanted classes."""
    boxes = []
    if path.exists():
        for line in path.read_text().splitlines():
            v = line.split()
            if len(v) < 5 or int(v[0]) not in classes:
                continue
            n = [float(x) for x in v[1:]]
            if len(n) == 4:
                boxes.append((n[0] - n[2] / 2, n[1] - n[3] / 2, n[0] + n[2] / 2, n[1] + n[3] / 2))
            else:
                boxes.append((min(n[0::2]), min(n[1::2]), max(n[0::2]), max(n[1::2])))
    return boxes


def yolo_line(c, b):
    return f"{c} {(b[0] + b[2]) / 2:.6f} {(b[1] + b[3]) / 2:.6f} {b[2] - b[0]:.6f} {b[3] - b[1]:.6f}"


# ---------- which photo goes where ----------

def recording(path):
    """Vairão frame '163311_frame_0004.png' -> recording '163311'."""
    return Path(path).stem.split("_frame")[0]


def vairao_split(path):
    r = recording(path)
    return "valid" if r in VAI_VALID else "test_vairao" if r in VAI_TEST else "train"


def lm_quota(n_other, share=LM_SHARE):
    """How many label map photos keep label map at `share` of the training photos."""
    return round(share / (1 - share) * n_other)


def escayard_blocks():
    blocks = {"B7": [], "B9": []}
    with open(ESCA / "datasheet.csv", encoding="utf-8-sig") as f:
        for r in csv.DictReader(f):
            if r["Vineyard"].strip() in blocks and r["File"].strip():
                blocks[r["Vineyard"].strip()].append(r["File"].strip())
    photos = {p.name: p for p in (ESCA / "photos").rglob("*.jpg")}
    return {b: [photos[n] for n in sorted(names) if n in photos] for b, names in blocks.items()}


def keep_pseudo(found, found_mirror):
    """Auto-labels for one photo, or None when the photo must be left out.
    found / found_mirror: [(box, conf)] from the photo and from its mirror image (boxes already mirrored back).
    Kept: boxes with conf >= SURE that are found again on the mirror (IoU >= 0.6, conf >= MIRROR) and at least 8% tall.
    Left out: photos with any unsure box (between UNSURE and SURE, or sure but not confirmed), or with no trunk."""
    kept = []
    for box, conf in found:
        if conf < UNSURE:
            continue
        confirmed = any(iou(box, m) >= 0.6 and c >= MIRROR for m, c in found_mirror)
        if conf < SURE or not confirmed:
            return None
        if box[3] - box[1] >= 0.08:
            kept.append(box)
    return kept or None


def posts_for(candidates, trunks):
    """YOLOE "post" boxes [(box, conf, prompt)] -> post boxes that are tall and do not overlap a trunk."""
    out = []
    for box, conf, prompt in candidates:
        tall = (box[3] - box[1]) >= 4 * (box[2] - box[0])
        if prompt in ("wooden post", "metal pole", "concrete post") and conf >= POST_CONF and tall and \
                all(iou(box, t) < 0.1 and inside(box, t) < 0.3 for t in trunks):
            out.append(box)
    return out


# ---------- models (imported only when needed, so the maths above runs without ultralytics) ----------

def load_image(path):
    import cv2
    img = cv2.imread(str(path))
    h, w = img.shape[:2]
    s = MAX_SIDE / max(h, w)
    return cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA) if s < 1 else img


def predict(model, img, imgsz, conf, cls=0):
    """[(normalized box, conf)] of one class."""
    h, w = img.shape[:2]
    r = model.predict(img, imgsz=imgsz, conf=conf, verbose=False)[0]
    return [((b[0] / w, b[1] / h, b[2] / w, b[3] / h), c)
            for b, c, k in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist(), r.boxes.cls.tolist()) if int(k) == cls]


class PostLabeler:
    def __init__(self):
        from ultralytics import YOLOE
        local = ROOT / "models" / "yoloe-11l-seg.pt"
        self.m = YOLOE(str(local) if local.exists() else "yoloe-11l-seg.pt")  # Ultralytics downloads it if missing
        self.m.set_classes(POST_PROMPTS, self.m.get_text_pe(POST_PROMPTS))

    def __call__(self, img, trunks):
        h, w = img.shape[:2]
        r = self.m.predict(img, imgsz=1024, conf=POST_CONF, verbose=False)[0]
        cands = [((b[0] / w, b[1] / h, b[2] / w, b[3] / h), c, POST_PROMPTS[int(k)])
                 for b, c, k in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist(), r.boxes.cls.tolist())]
        return posts_for(cands, trunks)


# ---------- build ----------

def write(split, name, img, trunks, posts=()):
    import cv2
    for sub in ("images", "labels"):
        (DS / split / sub).mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(DS / split / "images" / f"{name}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 92])
    lines = [yolo_line(0, b) for b in trunks] + [yolo_line(1, b) for b in posts]
    (DS / split / "labels" / f"{name}.txt").write_text("\n".join(lines))


def build(args):
    import cv2
    from ultralytics import YOLO

    if DS.exists():
        shutil.rmtree(DS)
    manual = json.loads(MANUAL.read_text())
    items = []  # (split, name, photo path, trunk boxes or None = auto-label with v2)
    lm_train = sorted((LM / "train" / "images").glob("*.jpg"))
    items += [("test_lm", f"lm_{p.stem}", p, read_labels(LM / "test" / "labels" / f"{p.stem}.txt", {2, 3}))
              for p in sorted((LM / "test" / "images").glob("*.jpg"))]
    for folder in (HUM / "train" / "Original_photos", HUM / "valid"):
        items += [("train", f"hu_{p.stem}", p, read_labels(folder / f"{p.stem}.txt", {0})) for p in sorted(folder.glob("*.jpg"))]
    for p in sorted(VAI.rglob("*.png")):
        items.append((vairao_split(p), f"va_{p.stem}", p, read_labels(p.parent.parent / "labels" / f"{p.stem}.txt", {0})))
    b7 = escayard_blocks()["B7"]
    items += [("valid", f"esm_{p.stem}", p, [tuple(b) for b in manual[p.name]]) for p in b7 if p.name in manual]
    items += [("train", f"es_{p.stem}", p, None) for p in b7 if p.name not in manual]
    mildew = sorted(MILDEW.glob("*.jpg"))
    items += [("train", f"neg_{p.stem}", p, []) for p in mildew[0::2]]  # foliage without trunks: negatives
    items += [("test_foliage", f"neg_{p.stem}", p, []) for p in mildew[1::2]]

    n_other = sum(s == "train" for s, *_ in items)
    quota = lm_quota(n_other)
    random.Random(0).shuffle(lm_train)
    items += [("train", f"lm_{p.stem}", p, read_labels(LM / "train" / "labels" / f"{p.stem}.txt", {2, 3}))
              for p in lm_train[:quota]]
    print(f"label map: {min(quota, len(lm_train))} of {len(lm_train)} training photos kept "
          f"({LM_SHARE:.0%} of the training photos)", flush=True)

    teacher = YOLO(str(V2))
    posts = PostLabeler()
    stats = {"posts": 0, "pseudo_kept": 0, "pseudo_left_out": 0}
    t0 = time.time()
    for i, (split, name, path, trunks) in enumerate(items, 1):
        img = load_image(path)
        if trunks is None:  # EscaYard B7 without hand marks: auto-label with v2 + mirror check
            found = predict(teacher, img, 1280, UNSURE)
            mirror = [((1 - b[2], b[1], 1 - b[0], b[3]), c) for b, c in predict(teacher, cv2.flip(img, 1), 1280, MIRROR)]
            trunks = keep_pseudo(found, mirror)
            if trunks is None:
                stats["pseudo_left_out"] += 1
                continue
            stats["pseudo_kept"] += 1
        found_posts = posts(img, trunks) if split in ("train", "valid") else []
        stats["posts"] += len(found_posts)
        write(split, name, img, trunks, found_posts)
        if i % 100 == 0 or i == len(items):
            left = (time.time() - t0) / i * (len(items) - i)
            print(f"{i}/{len(items)} photos ({i / len(items):.0%}) | about {left / 60:.0f} min left", flush=True)

    (DS / "data.yaml").write_text(f"path: {DS.as_posix()}\ntrain: train/images\nval: valid/images\n"
                                  "names:\n  0: trunchi\n  1: stalp\n", encoding="utf-8")
    counts = {}
    for f in (DS / "train" / "images").glob("*.jpg"):
        counts[f.stem.split("_")[0]] = counts.get(f.stem.split("_")[0], 0) + 1
    print("Training photos per source:", counts, "| label map share:",
          f"{counts.get('lm', 0) / max(1, sum(counts.values())):.0%}")
    print({s: len(list((DS / s / "images").glob("*.jpg"))) for s in ("train", "valid") + TEST_SETS})
    print(f"EscaYard B7 auto-labels: {stats['pseudo_kept']} photos kept, {stats['pseudo_left_out']} left out (unsure) | "
          f"posts auto-labeled: {stats['posts']}")


# ---------- train ----------

def train(args):
    from ultralytics import YOLO

    t0 = time.time()

    def print_epoch(trainer):
        ep, total = trainer.epoch + 1, trainer.epochs
        if ep > total:  # the final validation after training calls this once more
            return
        elapsed = time.time() - t0
        best = getattr(getattr(trainer, "stopper", None), "best_epoch", ep)  # already counts from 1
        print(f"==> Epoch {ep}/{total} ({ep / total:.0%}) | validation mAP50: "
              f"{trainer.metrics.get('metrics/mAP50(B)', 0.0):.3f} | best so far: epoch {best} | "
              f"{elapsed / 60:.0f} min | left: max ~{elapsed / ep * (total - ep) / 60:.0f} min", flush=True)

    m = YOLO(args.init)
    m.add_callback("on_fit_epoch_end", print_epoch)
    # AdamW with an explicit learning rate: optimizer="auto" picks its own and ignores lr0 on short fine-tunes
    m.train(data=str(DS / "data.yaml"), imgsz=args.imgsz, epochs=args.epochs, patience=args.patience, batch=args.batch,
            optimizer="AdamW", lr0=args.lr0, save_period=1, workers=2, project=str(RUNS), name=args.name,
            exist_ok=True, plots=False, hsv_h=0.03, hsv_s=0.8, hsv_v=0.6, degrees=5, scale=0.6, fraction=args.fraction,
            amp=not args.no_amp)


# ---------- evaluation ----------

def score_photos(model, folder, imgsz, conf):
    """[(found boxes with conf, real trunk boxes)] for every photo of a dataset folder."""
    out = []
    for p in sorted((folder / "images").glob("*.jpg")):
        import cv2
        img = cv2.imread(str(p))
        out.append((predict(model, img, imgsz, conf), read_labels(folder / "labels" / f"{p.stem}.txt", {0})))
    return out


def prf(per_photo, conf, min_iou):
    """Precision, recall, F1 at a threshold: each real trunk matched at most once, most confident box first."""
    right = found = real = 0
    for dets, gt in per_photo:
        free = list(gt)
        for box, c in sorted(dets, key=lambda d: -d[1]):
            if c < conf:
                continue
            found += 1
            best = max(free, key=lambda g: iou(g, box), default=None)
            if best is not None and iou(best, box) >= min_iou:
                free.remove(best)
                right += 1
        real += len(gt)
    p = right / found if found else 0.0
    r = right / real if real else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0), right, found, real


def best_threshold(per_photo, min_iou=0.3, thresholds=(0.2, 0.25, 0.3, 0.35, 0.4, 0.5, 0.6)):
    """(F1, threshold) with the highest F1."""
    return max((prf(per_photo, t, min_iou)[2], t) for t in thresholds)


def select(args):
    from ultralytics import YOLO
    w = RUNS / args.run / "weights"
    cands = sorted(w.glob("epoch*.pt"), key=lambda p: int(p.stem[5:])) + [w / "best.pt", w / "last.pt"]
    cands = [c for c in cands if c.exists()] + [V2]
    table = []
    for i, c in enumerate(cands, 1):
        f1, t = best_threshold(score_photos(YOLO(str(c)), DS / "valid", args.imgsz, 0.2))
        table.append((f1, t, c))
        print(f"{i}/{len(cands)} ({i / len(cands):.0%}) {c.name:14s} F1 {f1:.3f} at conf {t}", flush=True)
    f1, t, best = max(r for r in table if r[2] != V2)
    v2 = next(r for r in table if r[2] == V2)
    print(f"\nBest: {best.name} (F1 {f1:.3f} at conf {t}) | v2 on the same photos: F1 {v2[0]:.3f} at conf {v2[1]}")
    shutil.copy(best, RUNS / "model_trunchi_v3.pt")
    (RUNS / "v3_choice.json").write_text(json.dumps({"epoch": best.name, "conf": t, "f1": f1, "v2_f1": v2[0]}))
    print("Saved:", RUNS / "model_trunchi_v3.pt")


def b9_photos_with_trunk(model, imgsz, conf):
    b9 = escayard_blocks()["B9"]
    return sum(bool(predict(model, load_image(p), imgsz, conf)) for p in b9), len(b9)


def report(args):
    from ultralytics import YOLO
    rows = []
    for name, weights, conf in (("v2 (now)", V2, 0.3), ("v3", Path(args.weights), args.conf)):
        m = YOLO(str(weights))
        r = {"model": name, "conf": conf}
        for s in ("test_vairao", "test_lm"):
            per = score_photos(m, DS / s, args.imgsz, conf)
            for min_iou in (0.3, 0.5):
                p, rec, f1, right, found, real = prf(per, conf, min_iou)
                r[f"{s} IoU{min_iou} P/R/F1"] = f"{p:.2f}/{rec:.2f}/{f1:.2f}"
            r[f"{s} photos with trunk found"] = f"{sum(bool(d) for d, g in per if g)}/{sum(bool(g) for _, g in per)}"
        fol = score_photos(m, DS / "test_foliage", args.imgsz, conf)
        r["foliage false alarms (photos)"] = f"{sum(bool(d) for d, _ in fol)}/{len(fol)}"
        k, n = b9_photos_with_trunk(m, args.imgsz, conf)
        r["EscaYard B9 photos with trunk"] = f"{k}/{n}"
        rows.append(r)
        print(f"{name} done", flush=True)
    keys = [k for k in rows[0] if k != "model"]
    print(f"\n{'':40s}{rows[0]['model']:>18s}{rows[1]['model']:>18s}")
    for k in keys:
        print(f"{k:40s}{str(rows[0][k]):>18s}{str(rows[1][k]):>18s}")
    (RUNS / "v3_report.json").write_text(json.dumps(rows, indent=1))


def main():
    p = argparse.ArgumentParser()
    sub = p.add_subparsers(dest="cmd", required=True)
    sub.add_parser("build")
    t = sub.add_parser("train")
    t.add_argument("--name", default="v3")
    t.add_argument("--init", default=str(V2))
    t.add_argument("--imgsz", type=int, default=960)
    t.add_argument("--epochs", type=int, default=30)
    t.add_argument("--patience", type=int, default=10)
    t.add_argument("--batch", type=int, default=8)
    t.add_argument("--lr0", type=float, default=0.0005)
    t.add_argument("--fraction", type=float, default=1.0, help="share of the training photos (e.g. 0.05 for a quick check)")
    t.add_argument("--no-amp", action="store_true", help="no mixed precision (GTX 16xx cards misbehave with it)")
    s = sub.add_parser("select")
    s.add_argument("--run", default="v3")
    s.add_argument("--imgsz", type=int, default=1280)
    r = sub.add_parser("report")
    r.add_argument("--weights", required=True)
    r.add_argument("--conf", type=float, default=0.3)
    r.add_argument("--imgsz", type=int, default=1280)
    args = p.parse_args()
    {"build": build, "train": train, "select": select, "report": report}[args.cmd](args)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/tv/data/manual_trunks_b7.json
{"IMG_9457.jpg":[[0.1654,0.8322,0.2527,0.994],[0.8653,0.7092,0.9169,0.8001]],"IMG_20220714_214202.jpg":[[0.276,0.7147,0.3849,1],[0.8096,0.4886,0.8712,0.6017]],"IMG_20220714_212249.jpg":[[0.1488,0.6726,0.2352,0.9652]],"IMG_9479.jpg":[[0.3175,0.6504,0.4771,0.963],[0.0989,0.5174,0.1721,0.6394]],"IMG_20220714_213134.jpg":[[0.064,0.5208,0.256,0.9264],[0.8146,0.5341,0.9027,0.707]],"IMG_20220714_214717.jpg":[[0.1147,0.6105,0.2685,0.9054]],"IMG_20220714_212835.jpg":[[0.1995,0.7302,0.2976,0.9818]],"IMG_9455.jpg":[[0.6309,0.1805,0.6941,0.3678],[0.4838,0.1384,0.5395,0.2503],[0.4347,0.1062,0.4755,0.1805]],"IMG_20220714_212515.jpg":[[0.2552,0.7347,0.335,0.9896]],"IMG_20220714_214637.jpg":[[0.1538,0.6937,0.2328,0.9896]],"IMG_20220714_212558.jpg":[[0.2028,0.6637,0.2993,0.9974]],"IMG_20220714_212141.jpg":[[0.1214,0.6538,0.1729,0.932]],"IMG_20220714_212533.jpg":[[0.724,0.3689,0.7573,0.462],[0.3948,0.7845,0.4298,0.9209]],"IMG_9431.jpg":[[0.1588,0.7103,0.2469,0.9453],[0.8462,0.5995,0.9219,0.738]],"IMG_20220714_214536.jpg":[[0.6509,0.5551,0.6974,0.6238],[0.3466,0.8001,0.3915,0.8898]],"IMG_20220714_214047.jpg":[[0.2436,0.6892,0.3067,0.9419],[0.1114,0.4532,0.1388,0.5119]],"IMG_20220714_213914.jpg":[[0.0923,0.3955,0.1197,0.4565],[0.724,0.4332,0.7556,0.5041]],"IMG_9427.jpg":[[0.6301,0.421,0.66,0.5518],[0.266,0.4908,0.2959,0.595],[0.3092,0.4698,0.3333,0.5163],[0.0773,0.4731,0.0948,0.5263]],"IMG_9483.jpg":[[0.2959,0.165,0.3732,0.2991],[0.3932,0.6072,0.4281,0.6881]],"IMG_20220714_213458.jpg":[[0.0765,0.4432,0.1147,0.533],[0.3333,0.6926,0.3533,0.8377]]}

## 8. Setul de date (~20–30 min: etichete automate B7 + stâlpi)

In [ ]:
!cd /content && python trunk_v3.py build

## 9. Antrenare (~1–2 ore, progres pe epoci; fiecare epocă se salvează în Drive)

In [ ]:
!cd /content && python trunk_v3.py train --name v3 --epochs 30 --batch 8

### Continuare (doar dacă sesiunea s-a închis în timpul antrenării)
Rulați din nou celulele 1–8, apoi scoateți `#` din linia de mai jos: antrenarea continuă de unde a rămas.

In [ ]:
# !cd /content && python -c "from ultralytics import YOLO; YOLO('{R}/v3/weights/last.pt').train(resume=True)"

## 10. Alegem cea mai bună epocă pe via-țintă (~10 min)

In [ ]:
!cd /content && python trunk_v3.py select --run v3

## 11. Testul final: v2 vs v3 pe poze nefolosite niciodată la antrenare (~10 min)

In [ ]:
choice = json.load(open(f"{R}/v3_choice.json"))
print(choice)
conf = choice["conf"]
!cd /content && python trunk_v3.py report --weights "{R}/model_trunchi_v3.pt" --conf {conf}

## 12. Pe laptop
1. Descărcați din Drive `trunk_v3_runs/model_trunchi_v3.pt` în `models/`.
2. Spuneți-i lui Claude pragul ales (`v3_choice.json`) și tabelul de mai sus: îl verificăm și pe laptop
   (`python -m tools.evaluate_boxes` pe Vairão și `python -m tools.evaluate_escayard --vineyards B9`) și, dacă e mai bun,
   îl punem în `vineyard/config.py` în locul lui v2.